# HotPotQA - ReAct + CoT (LangGraph `StateGraph` 버전)

CoT(Chain-of-Thought)를 ReAct 에이전트 내부에 통합한다.  
CoT는 별도 시스템이 아니라, ReAct의 매 Thought 단계에서 항상 동작한다.

| 방식 | 설명 |
|------|------|
| ReAct | 기본 프롬프트, temperature=0 |
| ReAct-CoT | CoT 강화 프롬프트, temperature=0 |
| ReAct-CoT-SC | ReAct-CoT를 N번 샘플링 + 다수결 투표 |

## 1. Setup

In [ ]:
import os
import sys
import json
import random
import time

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from tools import search, lookup, wiki_state
from eval_utils import (
    load_hotpotqa, normalize_answer, f1_score, get_metrics, majority_vote
)

## 2. 데이터 & 프롬프트 로딩

In [ ]:
data = load_hotpotqa("dev")
print(f"Loaded {len(data)} questions.")

with open("../prompts/prompts_naive.json", "r", encoding="utf-8") as f:
    prompt_dict = json.load(f)

## 3. 프롬프트 정의

기본 ReAct 프롬프트와 CoT 강화 프롬프트를 비교한다.  
CoT 강화 프롬프트는 매 Thought에서 단계적 추론을 명시적으로 요구한다.

In [ ]:
# --- 기본 ReAct 프롬프트 ---
REACT_INSTRUCTION_BASE = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

Thought는 현재 상황에 대해 추론하는 단계이며,
Action은 다음 세 가지 유형 중 하나여야 한다:

(1) Search[entity]: Wikipedia에서 해당 entity를 검색.
(2) Lookup[keyword]: 현재 문서에서 keyword가 포함된 다음 문장을 반환.
(3) Finish[answer]: 최종 답을 반환하고 과제를 종료.

다음은 몇 가지 예시이다.
"""

# --- CoT 강화 ReAct 프롬프트 ---
REACT_INSTRUCTION_COT = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

각 Thought 단계에서는 반드시 단계적으로 추론(chain-of-thought)하시오:
1. 현재까지 파악한 사실을 정리한다.
2. 질문에 답하기 위해 아직 필요한 정보를 식별한다.
3. 다음 행동을 선택한 근거를 설명한다.

Action은 다음 세 가지 유형 중 하나여야 한다:

(1) Search[entity]: Wikipedia에서 해당 entity를 검색.
(2) Lookup[keyword]: 현재 문서에서 keyword가 포함된 다음 문장을 반환.
(3) Finish[answer]: 최종 답을 반환하고 과제를 종료.

다음은 몇 가지 예시이다.
"""

# few-shot 예시는 동일하게 webthink_simple6 사용
FEW_SHOT = prompt_dict["webthink_simple6"]

REACT_PROMPT_BASE = REACT_INSTRUCTION_BASE + FEW_SHOT
REACT_PROMPT_COT = REACT_INSTRUCTION_COT + FEW_SHOT

print(f"Base prompt: {len(REACT_PROMPT_BASE)} chars")
print(f"CoT prompt:  {len(REACT_PROMPT_COT)} chars")

## 4. LLM 정의

In [ ]:
llm_greedy = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=1000)
llm_sampling = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7, max_tokens=1000)

print("LLMs initialized.")

## 5. ReAct 그래프 팩토리

LLM과 프롬프트를 파라미터로 받아 ReAct 그래프를 생성하는 팩토리 함수.  
이를 통해 기본/CoT/CoT-SC 세 가지 버전을 생성한다.

In [ ]:
class AgentState(TypedDict):
    question: str
    prompt: str
    current_step: int
    max_steps: int
    answer: str
    done: bool
    trajectory: str
    n_calls: int
    n_badcalls: int


def build_react_graph(llm):
    """주어진 LLM으로 ReAct 그래프를 생성한다."""

    def reasoning_node(state: AgentState) -> dict:
        step = state["current_step"]
        prompt = state["prompt"]
        query = prompt + f"Thought {step}:"
        response = llm.invoke(query, stop=[f"\nObservation {step}:"])
        thought_action = response.content.strip()

        n_calls = state["n_calls"] + 1
        n_badcalls = state["n_badcalls"]

        try:
            thought, action = thought_action.split(f"\nAction {step}: ")
        except ValueError:
            n_badcalls += 1
            n_calls += 1
            thought = thought_action.split("\n")[0]
            action_response = llm.invoke(
                prompt + f"Thought {step}: {thought}\nAction {step}:",
                stop=["\n"],
            )
            action = action_response.content.strip()

        if action:
            action = action[0].lower() + action[1:]

        return {
            "prompt": prompt + f"Thought {step}: {thought}\nAction {step}: {action}\n",
            "trajectory": state["trajectory"] + f"Thought {step}: {thought}\nAction {step}: {action}\n",
            "n_calls": n_calls,
            "n_badcalls": n_badcalls,
        }

    def tool_exec_node(state: AgentState) -> dict:
        step = state["current_step"]
        trajectory = state["trajectory"]

        lines = trajectory.strip().split("\n")
        action_line = ""
        for line in reversed(lines):
            if line.startswith(f"Action {step}:"):
                action_line = line.split(": ", 1)[1] if ": " in line else ""
                break
        action = action_line.strip()

        if action.startswith("search[") and action.endswith("]"):
            entity = action[len("search["):-1]
            obs = search.invoke({"entity": entity})
        elif action.startswith("lookup[") and action.endswith("]"):
            keyword = action[len("lookup["):-1]
            obs = lookup.invoke({"keyword": keyword})
        elif action.startswith("finish[") and action.endswith("]"):
            answer = action[len("finish["):-1]
            obs_text = f"Observation {step}: Episode finished.\n"
            return {
                "answer": answer, "done": True,
                "prompt": state["prompt"] + obs_text,
                "trajectory": trajectory + obs_text,
                "current_step": step + 1,
            }
        else:
            obs = f"Invalid action: {action}"

        obs = obs.replace("\\n", "")
        obs_text = f"Observation {step}: {obs}\n"
        return {
            "prompt": state["prompt"] + obs_text,
            "trajectory": trajectory + obs_text,
            "current_step": step + 1,
        }

    def should_continue(state: AgentState) -> str:
        if state.get("done", False):
            return "end"
        if state["current_step"] > state["max_steps"]:
            return "end"
        return "continue"

    graph = StateGraph(AgentState)
    graph.add_node("reasoning", reasoning_node)
    graph.add_node("tool_exec", tool_exec_node)
    graph.add_edge(START, "reasoning")
    graph.add_edge("reasoning", "tool_exec")
    graph.add_conditional_edges(
        "tool_exec", should_continue, {"continue": "reasoning", "end": END}
    )
    return graph.compile()


# 3개의 그래프 생성
react_base = build_react_graph(llm_greedy)     # ReAct (baseline)
react_cot = build_react_graph(llm_greedy)       # ReAct-CoT (같은 LLM, 프롬프트만 다름)
react_cot_sc = build_react_graph(llm_sampling)  # ReAct-CoT-SC (샘플링 LLM)

print("3 ReAct graphs compiled.")

## 6. 실행 함수 정의

In [ ]:
def run_react(
    app, react_prompt: str, question: str,
    to_print: bool = False, max_steps: int = 7,
) -> tuple[str, dict]:
    """ReAct 그래프를 실행한다."""
    wiki_state.reset()
    initial_state = {
        "question": question,
        "prompt": react_prompt + f"Question: {question}\n",
        "current_step": 1,
        "max_steps": max_steps,
        "answer": "",
        "done": False,
        "trajectory": f"Question: {question}\n",
        "n_calls": 0,
        "n_badcalls": 0,
    }
    result = app.invoke(initial_state)

    answer = result["answer"]
    info = {
        "answer": answer,
        "steps": result["current_step"] - 1,
        "n_calls": result["n_calls"],
        "done": result["done"],
        "trajectory": result["trajectory"],
    }

    if to_print:
        print(f"Q: {question}")
        print(result["trajectory"])
        print(f"=> Answer: {answer}")

    return answer, info


def run_react_sc(
    app, react_prompt: str, question: str,
    n_samples: int = 5, to_print: bool = False,
) -> tuple[str, dict]:
    """ReAct 그래프를 N번 실행하고 다수결 투표로 최종 답을 결정한다."""
    answers = []
    for i in range(n_samples):
        try:
            answer, info = run_react(app, react_prompt, question)
        except Exception:
            answer = ""
        answers.append(answer)
        if to_print:
            print(f"  [Run {i+1}/{n_samples}] {answer}")

    valid_answers = [a for a in answers if a]
    if not valid_answers:
        return "", {"answer": "", "confidence": 0.0, "all_answers": answers}

    final_answer, confidence = majority_vote(valid_answers)

    if to_print:
        print(f"  Votes: {answers}")
        print(f"  => Final: {final_answer} (confidence: {confidence:.0%})")

    return final_answer, {
        "answer": final_answer,
        "confidence": confidence,
        "all_answers": answers,
    }


print("Run functions defined.")

## 7. 단일 질문 테스트 (3가지 방식 비교)

In [ ]:
N_SC_SAMPLES = 5

question, gt_answer = data[0]
print(f"Question: {question}")
print(f"GT: {gt_answer}")
print("=" * 60)

# 1) ReAct (baseline)
print("\n[1] ReAct (baseline)")
pred1, _ = run_react(react_base, REACT_PROMPT_BASE, question, to_print=True)
print(f"EM: {normalize_answer(pred1) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 2) ReAct-CoT
print("\n[2] ReAct-CoT")
pred2, _ = run_react(react_cot, REACT_PROMPT_COT, question, to_print=True)
print(f"EM: {normalize_answer(pred2) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 3) ReAct-CoT-SC
print(f"\n[3] ReAct-CoT-SC (n={N_SC_SAMPLES})")
pred3, info3 = run_react_sc(
    react_cot_sc, REACT_PROMPT_COT, question,
    n_samples=N_SC_SAMPLES, to_print=True,
)
print(f"EM: {normalize_answer(pred3) == normalize_answer(gt_answer)}")

## 8. 배치 비교 평가

In [ ]:
N_EVAL = 10
N_SC_SAMPLES = 5

idxs = list(range(len(data)))
random.Random(233).shuffle(idxs)

methods = {
    "ReAct": lambda q: run_react(react_base, REACT_PROMPT_BASE, q),
    "ReAct-CoT": lambda q: run_react(react_cot, REACT_PROMPT_COT, q),
    "ReAct-CoT-SC": lambda q: run_react_sc(react_cot_sc, REACT_PROMPT_COT, q, N_SC_SAMPLES),
}

all_results = {name: [] for name in methods}
old_time = time.time()

for i, idx in enumerate(idxs[:N_EVAL]):
    question, gt_answer = data[idx]
    print(f"\n[{i+1}/{N_EVAL}] Q: {question}")
    print(f"  GT: {gt_answer}")

    for method_name, method_fn in methods.items():
        try:
            pred, info = method_fn(question)
        except Exception as e:
            print(f"  {method_name}: ERROR - {e}")
            pred = ""

        metrics = get_metrics(pred, gt_answer)
        all_results[method_name].append(metrics)

        em_sum = sum(r["em"] for r in all_results[method_name])
        avg_em = em_sum / len(all_results[method_name])
        print(f"  {method_name:15s}: {pred[:50]:50s} | EM={avg_em:.3f} ({em_sum}/{len(all_results[method_name])})")

    elapsed = (time.time() - old_time) / (i + 1)
    print(f"  ({elapsed:.1f}s/q)")
    print("-" * 80)
    sys.stdout.flush()

## 9. 최종 결과 비교

In [ ]:
print(f"\n{'='*60}")
print(f"  Final Results ({N_EVAL} samples)")
print(f"  Agent: StateGraph, SC n={N_SC_SAMPLES}")
print(f"{'='*60}")
print(f"{'Method':20s} {'EM':>8s} {'F1':>8s}")
print(f"{'-'*20} {'-'*8} {'-'*8}")

for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    total_em = sum(r["em"] for r in results)
    total_f1 = sum(r["f1"] for r in results)
    print(f"{method_name:20s} {total_em/n:8.3f} {total_f1/n:8.3f}")

print(f"{'='*60}")

## 10. 결과 저장

In [ ]:
os.makedirs("trajs", exist_ok=True)

summary = {}
for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    summary[method_name] = {
        "n_samples": n,
        "em": sum(r["em"] for r in results) / n,
        "f1": sum(r["f1"] for r in results) / n,
    }

with open("trajs/react_cot_langgraph.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved to trajs/react_cot_langgraph.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))